# KNN - Titanic

Objetivo: entrenar un clasificador KNN y evaluar en entrenamiento y prueba.

## 1) Librerias

In [1]:
!pip install pandas matplotlib seaborn scikit-learn

In [2]:
# Datos
import pandas as pd


In [3]:
# Calculo numerico
import numpy as np


In [4]:
# Modelo KNN
from sklearn.neighbors import KNeighborsClassifier


In [5]:
# Metricas
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report


## 2) Cargar datos

In [6]:
# Leer archivo
titanic = pd.read_csv("titanic.csv")


In [7]:
# Vista rapida
titanic.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
# Tipos de columnas
titanic.info()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## 3) Preparar datos

In [9]:
# Convertir tipos
titanic["Survived"] = pd.to_numeric(titanic["Survived"], errors="coerce")


In [10]:
# Convertir tipos
titanic["Sex"] = titanic["Sex"].astype("category")


In [11]:
# Convertir tipos
titanic["Pclass"] = titanic["Pclass"].astype("category")


In [12]:
# Eliminar faltantes clave
datos_modelo = titanic.dropna(subset=["Age", "Embarked"])


In [13]:
# Seleccionar variables
datos_modelo = datos_modelo[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]


In [14]:
# Resumen
datos_modelo.describe(include="all")


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
count,712.000000,712.0,712,712.000000,712.000000,712.000000,712.000000,712
unique,NaN,3.0,2,NaN,NaN,NaN,NaN,3
top,NaN,3.0,male,NaN,NaN,NaN,NaN,S
freq,NaN,355.0,453,NaN,NaN,NaN,NaN,554
mean,0.404494,NaN,NaN,29.642093,0.514045,0.432584,34.567251,NaN
std,0.491139,NaN,NaN,14.492933,0.930692,0.854181,52.938648,NaN
min,0.000000,NaN,NaN,0.420000,0.000000,0.000000,0.000000,NaN
25%,0.000000,NaN,NaN,20.000000,0.000000,0.000000,8.050000,NaN
50%,0.000000,NaN,NaN,28.000000,0.000000,0.000000,15.645850,NaN
75%,1.000000,NaN,NaN,38.000000,1.000000,1.000000,33.000000,NaN


## 4) Separar entrenamiento y prueba (80/20)

In [15]:
# Muestreo reproducible
np.random.seed(123)


In [16]:
# Indices de entrenamiento
indices = np.random.choice(datos_modelo.index, size=int(0.8 * len(datos_modelo)), replace=False)


In [17]:
# Conjuntos
entrenamiento = datos_modelo.loc[indices].copy()
prueba = datos_modelo.drop(indices).copy()


## 5) Variables para KNN (numericas + codificacion simple)

In [18]:
# Entrenamiento para KNN
entrenamiento_knn = entrenamiento.copy()
entrenamiento_knn["Sex_num"] = entrenamiento_knn["Sex"].astype("category").cat.codes
entrenamiento_knn["Pclass_num"] = entrenamiento_knn["Pclass"].astype("category").cat.codes + 1
entrenamiento_knn = entrenamiento_knn[["Age", "SibSp", "Parch", "Fare", "Sex_num", "Pclass_num"]]


In [19]:
# Prueba para KNN
prueba_knn = prueba.copy()
prueba_knn["Sex_num"] = prueba_knn["Sex"].astype("category").cat.codes
prueba_knn["Pclass_num"] = prueba_knn["Pclass"].astype("category").cat.codes + 1
prueba_knn = prueba_knn[["Age", "SibSp", "Parch", "Fare", "Sex_num", "Pclass_num"]]


In [20]:
# Normalizar con min y max de entrenamiento
min_vals = entrenamiento_knn.min()
max_vals = entrenamiento_knn.max()
rango = max_vals - min_vals
rango[rango == 0] = 1
entrenamiento_knn_norm = (entrenamiento_knn - min_vals) / rango
prueba_knn_norm = (prueba_knn - min_vals) / rango


## 6) Entrenar y predecir

In [21]:
# Crear modelo KNN
modelo = KNeighborsClassifier(n_neighbors=5)


In [22]:
# Ajustar modelo
modelo.fit(entrenamiento_knn_norm, entrenamiento["Survived"])


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [23]:
# Prediccion entrenamiento
pred_train = modelo.predict(entrenamiento_knn_norm)


In [24]:
# Prediccion prueba
pred_test = modelo.predict(prueba_knn_norm)


## 7) Evaluacion

In [25]:
# Matriz de confusion entrenamiento
cm = confusion_matrix(entrenamiento["Survived"], pred_train)
pd.DataFrame(cm, index=['Real_0', 'Real_1'], columns=['Pred_0', 'Pred_1'])


,Pred_0,Pred_1
Real_0,309,32
Real_1,49,179


In [26]:
# Reporte de clasificacion
print(classification_report(entrenamiento["Survived"], pred_train, digits=4, zero_division=0))


              precision    recall  f1-score   support

           0     0.8631    0.9062    0.8841       341
           1     0.8483    0.7851    0.8155       228

    accuracy                         0.8576       569
   macro avg     0.8557    0.8456    0.8498       569
weighted avg     0.8572    0.8576    0.8566       569



In [27]:
# Matriz de confusion prueba
cm = confusion_matrix(prueba["Survived"], pred_test)
pd.DataFrame(cm, index=['Real_0', 'Real_1'], columns=['Pred_0', 'Pred_1'])


,Pred_0,Pred_1
Real_0,70,13
Real_1,16,44


In [28]:
# Reporte de clasificacion
print(classification_report(prueba["Survived"], pred_test, digits=4, zero_division=0))


              precision    recall  f1-score   support

           0     0.8140    0.8434    0.8284        83
           1     0.7719    0.7333    0.7521        60

    accuracy                         0.7972       143
   macro avg     0.7929    0.7884    0.7903       143
weighted avg     0.7963    0.7972    0.7964       143



In [29]:
# Accuracy entrenamiento
accuracy_score(entrenamiento["Survived"], pred_train)


0.8576449912126538

In [30]:
# Precision entrenamiento
precision_score(entrenamiento["Survived"], pred_train, zero_division=0)


0.8483412322274881

In [31]:
# Recall entrenamiento
recall_score(entrenamiento["Survived"], pred_train, zero_division=0)


0.7850877192982456

In [32]:
# Accuracy prueba
accuracy_score(prueba["Survived"], pred_test)


0.7972027972027972

In [33]:
# Precision prueba
precision_score(prueba["Survived"], pred_test, zero_division=0)


0.7719298245614035

In [34]:
# Recall prueba
recall_score(prueba["Survived"], pred_test, zero_division=0)


0.7333333333333333

In [35]:
# Tabla resumen
resumen_metricas = pd.DataFrame({
    "Metrica": ["Accuracy", "Precision", "Recall"],
    "Entrenamiento": [
        accuracy_score(entrenamiento["Survived"], pred_train),
        precision_score(entrenamiento["Survived"], pred_train, zero_division=0),
        recall_score(entrenamiento["Survived"], pred_train, zero_division=0),
    ],
    "Prueba": [
        accuracy_score(prueba["Survived"], pred_test),
        precision_score(prueba["Survived"], pred_test, zero_division=0),
        recall_score(prueba["Survived"], pred_test, zero_division=0),
    ]
})
resumen_metricas


,Metrica,Entrenamiento,Prueba
0,Accuracy,0.857645,0.797203
1,Precision,0.848341,0.771930
2,Recall,0.785088,0.733333
